# 4.1 — Creating a Custom Dataset

In [ ]:
import torch
from torch.utils.data import Dataset  
from PIL import Image 
import pandas as pd
from torchvision import transforms 

# 4.2 — Loading Training and Validation Data

In [2]:
train_df = pd.read_csv("../data_splits/train.csv")
val_df = pd.read_csv("../data_splits/val.csv")

print("Train:", len(train_df))
print("Validation:", len(val_df))

Train: 3094
Validation: 774


# 4.3 — Definition of the Dataset class

In [3]:
class TrafficVehicleDataset(Dataset):
    def __init__(self, dataframe, class_to_idx, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)  

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = row['path']
        label_name = row['label']

        image = Image.open(image_path).convert('RGB')
        label = self.class_to_idx[label_name]

        if self.transform is not None:
            image = self.transform(image)
        return image , label

# 4.4 — Creating class_to_idx (mapping classes to numbers)

In [4]:
classes = [
    'ambulance',
    'autobus',
    'kamyun',
    'minibus',
    'savari',
    'taxi',
    'vanet'
]
class_to_idx = {
    class_name : idx
    for idx , class_name in enumerate(classes)
}
print(class_to_idx)

{'ambulance': 0, 'autobus': 1, 'kamyun': 2, 'minibus': 3, 'savari': 4, 'taxi': 5, 'vanet': 6}


# 4.5 — Creating the First Dataset

In [5]:
train_dataset = TrafficVehicleDataset(
    dataframe=train_df,
    class_to_idx=class_to_idx,
    transform=None
)
val_dataset = TrafficVehicleDataset(
    dataframe=val_df,
    class_to_idx=class_to_idx,
    transform=None
)
print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))

Train dataset size: 3094
Validation dataset size: 774


# 4.6 — Testing a sample

In [6]:
image, label = train_dataset[0]

print("Image type:", type(image))
print("Label:", label)
print("Image size:", image.size)
print("Label type:", type(label))

Image type: <class 'PIL.Image.Image'>
Label: 0
Image size: (207, 276)
Label type: <class 'int'>


# 4.7 — Creating the First Transform

In [7]:
resize_transform = transforms.Resize((224, 224))
resized_image = resize_transform(image)

print("Original size:", image.size)
print("Resized size:", resized_image.size)

Original size: (207, 276)
Resized size: (224, 224)


In [8]:
to_tensor = transforms.ToTensor()
tensor_image = to_tensor(resized_image)

print("Type:", type(tensor_image))
print("Shape:", tensor_image.shape)
print("Min:", tensor_image.min())
print("Max:", tensor_image.max())

Type: <class 'torch.Tensor'>
Shape: torch.Size([3, 224, 224])
Min: tensor(0.0118)
Max: tensor(1.)


In [10]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 4.8 — Connecting a Transform to a Dataset

In [11]:
train_dataset = TrafficVehicleDataset(
    train_df,
    class_to_idx,
    transform=train_transform
)
val_dataset = TrafficVehicleDataset(
    val_df,
    class_to_idx,
    transform=val_transform
)

In [12]:
image, label = train_dataset[0]

print("Image type:", type(image))
print("Image shape:", image.shape)
print("Label:", label)
print("Label type:", type(label))

Image type: <class 'torch.Tensor'>
Image shape: torch.Size([3, 224, 224])
Label: 0
Label type: <class 'int'>


# The images is now ready for CNN.